In [0]:
%run ./materialization_utils/materialization_utils

In [0]:
import pyspark.sql.functions as F
import importlib
from datetime import datetime
from dateutil.relativedelta import relativedelta
import json
import ast
import logging
import gc
import os
import sys
import hashlib
from collections import defaultdict
from databricks.sdk.runtime import dbutils

build_id = dbutils.widgets.get("build_id")

In [0]:
#TODO: da parametrizzare
sys_path = '/Workspace/AI/creri/creri_model/files'
notebook_path = os.path.dirname(
    sys_path
)
sys.path.append("../..")

In [0]:
notebook_path = "/Workspace/" + os.path.dirname(
    dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
)
sys.path.append("..")
os.chdir(notebook_path)

In [0]:
def generate_month_starts(start_date: str, end_date: str) -> list[str]:
    """
    Genera una lista di tutte le date comprese tra start_date e end_date,
    corrispondenti al primo giorno di ogni mese.
    
    Parametri:
        start_date (str): data iniziale nel formato 'YYYY-MM-DD'
        end_date (str): data finale nel formato 'YYYY-MM-DD'
        
    Ritorna:
        list[str]: elenco di date (stringhe) in formato 'YYYY-MM-DD'
    """
    start = datetime.strptime(start_date, "%Y-%m-%d")
    end = datetime.strptime(end_date, "%Y-%m-%d")

    start = start.replace(day=1)
    
    dates = []
    while start <= end:
        dates.append(start.strftime("%Y-%m-%d"))
        start += relativedelta(months=1)

    return dates

def filter_dates_older_than_n_months(dates: list[str], n_months:str) -> list[str]:
    """
    Rimuove tutte le date comprese entro i 15 mesi precedenti a oggi.
    Mantiene solo quelle più vecchie di 15 mesi fa.
    
    Parametri:
        dates (list[str]): elenco di date in formato 'YYYY-MM-DD'
    
    Ritorna:
        list[str]: elenco filtrato di date più vecchie di 15 mesi
    """
    today = datetime.today()
    cutoff_date = today - relativedelta(months=int(n_months))
    
    filtered = [
        d for d in dates
        if datetime.strptime(d, "%Y-%m-%d") < cutoff_date
    ]
    
    return filtered


def load_yaml(path: str):
    """Load YAML file"""
    with open(path, "r") as f:
        return yaml.safe_load(f)


def build_materialization_plan(catalog, schema, metrics_input_params):
    """Genera un piano di materializzazione prendendo in considerazione le configurazioni lette aggiungendo le snapshot_date da materializzare classificate per gruppo di metriche"""
    
    plan = []
    
    for metric in metrics_input_params:
        # Leggi e prepara i dati direttamente come dizionari
        date_list = (
            spark.read.table(f"{catalog}.{schema}.{metric['name']}")
            .select("snapshot_date", "version")
            .distinct()
            .collect()
        )
        
        # Usa list comprehension invece del ciclo for
        plan.extend([
            {
                "group": "metrics",
                "metric": metric,
                "snapshot_date": obj["snapshot_date"],
                "version": obj["version"]
            }
            for obj in date_list
        ])
    
    return plan

In [0]:
cod_productName = "creri" #dbutils.widgets.get("cod_productName")

general_utils = GeneralUtils(spark)
metric_materializer = MetricMaterializer(spark)
logger = general_utils.set_logging()
config = general_utils.load_config(general_utils.get_variable(cod_productName)['config_setup_path'])
met_version_deployed = general_utils.load_config(general_utils.get_variable(cod_productName)['databricks_asset_path'])['variables']['metrics_version']['default']

local_adb_host = "https://" + spark.conf.get("spark.databricks.workspaceUrl") + "/"
env = [x for x in general_utils.load_config(general_utils.get_variable(cod_productName)['databricks_asset_path'])['targets'] if general_utils.load_config(general_utils.get_variable(cod_productName)['databricks_asset_path'])['targets'][x]['workspace']['host'] == local_adb_host][0]


catalog_name = config['environment'][env]['catalog']
schema_name = config['environment'][env]['schema']

try:
    metrics_list          = general_utils.get_list_configs(config['metrics'])
except:
    metrics_list = []
try:
    targets_list          = general_utils.get_list_configs(config['targets'])
except:
    targets_list = []
try:
    customer_bases_list   = general_utils.get_list_configs(config['customer_bases'])
except:
    customer_bases_list = []
try:
    master_training_list  = general_utils.get_list_configs(config['master_training'])
except:
    master_training_list = []
try:
    master_inference_list  = general_utils.get_list_configs(config['master_inference'])
except:
    master_inference_list = []
try:
    master_training_final_list = general_utils.get_list_configs(config['master_training_final'])
except:
    master_training_final_list = []

metrics_input_params = metrics_list + targets_list + customer_bases_list + master_training_list + master_inference_list + master_training_final_list
metrics_input_params

version_obj = []

In [0]:
def get_current_data_version(catalog_name: str, cod_productName: str, table_name: str, src_path: str) -> list:

    return spark.read.table(f"{catalog_name}.metadata.md_ai_dataset_versioning") \
        .select("cod_tableName", "version", "met_version") \
        .where(f"cod_productName = '{cod_productName}' and cod_tableName = '{table_name}' and src ='{src_path}' and flg_lastSnap = 1") \
        .collect()

def generate_write_data_version(metadata_obj, met_version_deployed, src_path, input_params_json, cod_productName, build_id, metric_version, table_name):
    if len(metadata_obj) > 0 and (metadata_obj[0]["met_version"] != met_version_deployed or str(metadata_obj[0]["version"]) != str(metric_version)):
        logger.info(f"UPDATEING new version on metadata layer")
        general_utils.populate_versioning_metadata(catalog_name, "metadata", "md_ai_dataset_versioning", cod_productName, table_name, src_path) 
        lg_ds_versioning_data = general_utils.get_ds_versioning_data(
            id_deploy=int(build_id),
            cod_productName=cod_productName,
            cod_tableName=table_name,
            version=str(metric_version),
            met_version=met_version_deployed,
            src=src_path,
            primary_key=str(input_params_json['primary_key']),
            fields_to_mask=str(input_params_json['fields_to_mask']),
            description=input_params_json['description'],
            certified=input_params_json.get('certified', None),
            dat_systemDate=datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            dat_startDate=datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            dat_endDate="9999" + datetime.now().strftime("%Y-%m-%d %H:%M:%S")[4:],
            flg_lastSnap=int(1)
        )

        df_lg_ds_versioning = general_utils.get_dataset_versioning_metadata(catalog_name, "metadata", "md_ai_dataset_versioning", lg_ds_versioning_data)
        general_utils.write_table(catalog_name, "metadata", "md_ai_dataset_versioning", df_lg_ds_versioning, "append")
        logger.info(f"New version for metric: {input_params_json['name']}")

    elif len(metadata_obj) == 0:
        logger.info(f"INSERTING new version on metadata layer")
        lg_ds_versioning_data = general_utils.get_ds_versioning_data(
            id_deploy=int(build_id),
            cod_productName=cod_productName,
            cod_tableName=table_name,
            version=str(metric_version),
            met_version=met_version_deployed,
            src=src_path,
            primary_key=str(input_params_json['primary_key']),
            fields_to_mask=str(input_params_json['fields_to_mask']),
            description=input_params_json['description'],
            certified=input_params_json.get('certified', None),
            dat_systemDate=datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            dat_startDate=datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            dat_endDate="9999" + datetime.now().strftime("%Y-%m-%d %H:%M:%S")[4:],
            flg_lastSnap=int(1)
        )

        df_lg_ds_versioning = general_utils.get_dataset_versioning_metadata(catalog_name, "metadata", "md_ai_dataset_versioning", lg_ds_versioning_data)
        general_utils.write_table(catalog_name, "metadata", "md_ai_dataset_versioning", df_lg_ds_versioning, "append")
        logger.info(f"New version for metric: {input_params_json['name']}")
    else:
        logger.info(f"No new version for metric: {input_params_json['name']}")
    
    return f"End writing process for metric: {input_params_json['name']}"

In [0]:
###############################################################################################
###############################################################################################
###################################### CERTIFIED METRICS ######################################
###############################################################################################
###############################################################################################
for input_params_json in metrics_input_params:
    if input_params_json['type'] == 'PREP_METRICS' and input_params_json['certified']:
        catalog_name = config['environment'][env]['catalog']
        schema_name = config['environment'][env]['schema']
        table_name = input_params_json['name']
        src_path = input_params_json['name'] + "." + input_params_json['class_name']
        logger.info(f"START process analysis for: {input_params_json['name']}")
        logger.info(f"Metric certified: {input_params_json['certified']}")

        try:
            logger.info(f"Reading source: {input_params_json['src']}")
            module = importlib.import_module(input_params_json['src'])
            metrica_class = getattr(module, input_params_json['class_name'])
            class_obj = metrica_class()

            metric_version = class_obj.get_version()  # versione della metrica
            logger.info(f"Metric version: {metric_version}")

            version_obj.append({input_params_json['class_name']: metric_version})

            # read/update metadata table
            metadata_obj = get_current_data_version(catalog_name, cod_productName, table_name, src_path)
            logger.info(f"get current version on delta table: {metadata_obj}")

            generate_write_data_version(metadata_obj, met_version_deployed, src_path, input_params_json, cod_productName, build_id, str(metric_version), table_name)

        except Exception as e:
            logger.info("Error on materialization: " + str(e))
            raise
        finally:
            gc.collect()

In [0]:
###############################################################################################
###############################################################################################
#################################### NOT CERTIFIED METRICS ####################################
###############################################################################################
###############################################################################################
for input_params_json in metrics_input_params:
    if input_params_json['type'] == 'PREP_METRICS' and not input_params_json['certified']:
        catalog_name = config['environment'][env]['catalog']
        schema_name = config['environment'][env]['schema']
        table_name = input_params_json['name']
        src_path = input_params_json['name'] + "." + input_params_json['class_name']
        logger.info(f"START process analysis for: {input_params_json['name']}")
        logger.info(f"Metric certified: {input_params_json['certified']}")

        try:
            logger.info(f"Reading source: {input_params_json['src']}")
            module = importlib.import_module(input_params_json['src'])
            metrica_class = getattr(module, input_params_json['class_name'])()

            metric_version = metrica_class.get_version()  # versione della metrica

            version_obj.append({input_params_json['class_name']: metric_version})

            metadata_obj = get_current_data_version(catalog_name, cod_productName, table_name, src_path)
            print(metadata_obj)
            generate_write_data_version(metadata_obj, met_version_deployed, src_path, input_params_json, cod_productName, build_id, str(metric_version), table_name)

        except Exception as e:
            logger.info("Error on materialization: " + str(e))
            raise
        finally:
            gc.collect()

In [0]:
############################################################################################
############################################################################################
############################################ CB ############################################
############################################################################################
############################################################################################
for input_params_json in metrics_input_params:  
    if 'PREP_CUSTOMER_BASE' in input_params_json['type']:
        catalog_name = config['environment'][env]['catalog']
        schema_name = config['environment'][env]['schema']
        table_name = input_params_json['name']
        src_path = input_params_json['name'] + "." + input_params_json['class_name']
        logger.info(f"START process analysis for: {input_params_json['name']}")
        logger.info(f"Metric certified: {input_params_json['certified']}")
        logger.info(f"Parameters: {input_params_json}")
        module = importlib.import_module(input_params_json['src'])
        cb_func = getattr(module, input_params_json['class_name'])()
        version_obj.append({input_params_json['class_name']: cb_func.get_version()})
        print("2: ", version_obj)
        # costruisco il dizionario metrica -> versione
        version_map = {
            key: value
            for d in version_obj
            for key, value in d.items()
        }
        # arricchisco src_metrics con la versione
        output = [f"{metric}_{version_map[metric]}" for metric in input_params_json['src_metrics']] + [f"{input_params_json['class_name']}_{cb_func.get_version()}"]
        print("3: ", output)
        result = ['|'.join(output)]
        print("4: ", result)

        metadata_obj = get_current_data_version(catalog_name, cod_productName, table_name, src_path)
        print("5: ", metadata_obj)
        generate_write_data_version(metadata_obj, met_version_deployed, src_path, input_params_json, cod_productName, build_id, str(result[0]), table_name)

In [0]:
#############################################################################################
#############################################################################################
########################################## MASTERS ##########################################
#############################################################################################
#############################################################################################
for input_params_json in metrics_input_params:
    if "PREP_MASTER_TRAINING" in input_params_json['type']:
        from src.masters.training_masters import *
        master_materilizer = masterMaterilizer(spark, logger)

        catalog_name = config['environment'][env]['catalog']
        schema_name = config['environment'][env]['schema']
        table_name = input_params_json['name']
        src_path = input_params_json['name'] + "." + input_params_json['class_name']
        logger.info(f"START process analysis for: {input_params_json['name']}")
        logger.info(f"Parameters: {input_params_json}")
        version_obj.append({input_params_json['class_name']: master_materilizer.get_version()})
        print("2: ", version_obj)

        # costruisco il dizionario metrica -> versione
        version_map = {
            key: value
            for d in version_obj
            for key, value in d.items()
        }
        # arricchisco src_metrics con la versione
        output = [f"{metric}_{version_map[metric]}" for metric in input_params_json['src_metrics']] + [f"{input_params_json['class_name']}_{master_materilizer.get_version()}"]
        print("3: ", output)
        result = ['|'.join(output)]
        print("4: ", result)

        metadata_obj = get_current_data_version(catalog_name, cod_productName, table_name, src_path)
        print("5: ", metadata_obj)
        generate_write_data_version(metadata_obj, met_version_deployed, src_path, input_params_json, cod_productName, build_id, str(result[0]), table_name)

    if "PREP_MASTER_INFERENCE" in input_params_json['type']:
        from src.masters.training_masters import *
        master_materilizer = masterMaterilizer(spark, logger)

        catalog_name = config['environment'][env]['catalog']
        schema_name = config['environment'][env]['schema']
        table_name = input_params_json['name']
        src_path = input_params_json['name'] + "." + input_params_json['class_name']
        logger.info(f"START process analysis for: {input_params_json['name']}")
        print("1: ", input_params_json)
        version_obj.append({input_params_json['class_name']: master_materilizer.get_version()})
        print("2: ", version_obj)

        # costruisco il dizionario metrica -> versione
        version_map = {
            key: value
            for d in version_obj
            for key, value in d.items()
        }
        # arricchisco src_metrics con la versione
        output = [f"{metric}_{version_map[metric]}" for metric in input_params_json['src_metrics']] + [f"{input_params_json['class_name']}_{cb_func.get_version()}"]
        print("3: ", output)
        result = ['|'.join(output)]
        print("4: ", result)

        metadata_obj = get_current_data_version(catalog_name, cod_productName, table_name, src_path)
        print("5: ", metadata_obj)
        generate_write_data_version(metadata_obj, met_version_deployed, src_path, input_params_json, cod_productName, build_id, str(result[0]), table_name)

In [0]:
###################################################################################################
###################################################################################################
########################################## FINAL MASTERS ##########################################
###################################################################################################
###################################################################################################
for input_params_json in metrics_input_params:
    if "PREP_MASTER_TRAINING_FINAL" in input_params_json['type']:
        
        from src.data_preparation.features_engineering import features_engineering, get_version

        catalog_name = config['environment'][env]['catalog']
        schema_name = config['environment'][env]['schema']
        table_name = input_params_json['name']
        src_path = input_params_json['name'] + "." + input_params_json['class_name']
        logger.info(f"START process analysis for: {input_params_json['name']}")
        print("1: ", input_params_json)
        version = get_version()
        version_obj.append({input_params_json['class_name']: version})
        print("2: ", version_obj)
        # costruisco il dizionario metrica -> versione
        version_map = {
            key: value
            for d in version_obj
            for key, value in d.items()
        }
        # arricchisco src_metrics con la versione
        output = [f"{metric}_{version_map[metric]}" for metric in input_params_json['src_metrics']] + [f"{input_params_json['class_name']}_{version}"]
        print("3: ", output)
        result = ['|'.join(output)]
        print("4: ", result)

        metadata_obj = get_current_data_version(catalog_name, cod_productName, table_name, src_path)
        print("5: ", metadata_obj)
        generate_write_data_version(metadata_obj, met_version_deployed, src_path, input_params_json, cod_productName, build_id, str(result[0]), table_name)

In [0]:
dbutils.notebook.exit("Success!!")

In [0]:
spark.sql("select * from p_ind_core_ai.metadata.md_ai_dataset_versioning order by dat_systemDate desc").display()


In [0]:
spark.sql("select snapshot_date, min(version) from p_ind_core_ai.creri.creri_met_accounttocodfiscalepiva where flg_lastSnap = 1 group by snapshot_date").display()

In [0]:
# %sql

# CREATE TABLE IF NOT EXISTS p_ind_core_ai.metadata.md_ai_dataset_versioning (
#     iid_recordID        BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 0 INCREMENT BY 1),
#     id_deploy           INT,
#     cod_productName     STRING,
#     cod_tableName       STRING,
#     version             STRING,
#     met_version    DOUBLE,
#     src                 STRING,
#     primary_key         STRING,
#     fields_to_mask      STRING,
#     description         STRING,
#     certified           STRING,
#     dat_systemDate      TIMESTAMP,
#     dat_startDate       TIMESTAMP,
#     dat_endDate         TIMESTAMP,
#     flg_lastSnap        INT
# );

In [0]:
spark.sql("select * from p_ind_core_ai.metadata.md_ai_dataset_versioning").printSchema()

In [0]:
spark.read.table("p_dpp_core_ai.creri.creri_train_mstr_credit_check_volture").write.mode("overwrite").option("mergeSchema", "true").option("overwriteSchema", "true").saveAsTable("p_ind_core_ai.creri.creri_train_mstr_credit_check_volture")